In [ ]:
import os
import shutil

source_folder = r"C:\Thesis\Data\sorted\test good"
destination_folder = r"C:\Thesis\Data\sorted\test_good_sorted"

os.makedirs(destination_folder, exist_ok=True)
files = sorted(os.listdir(source_folder))
index = 1

for i in range(0, len(files), 2): 
    if i + 1 < len(files):

        front_ext = os.path.splitext(files[i])[1]
        side_ext = os.path.splitext(files[i + 1])[1]
        new_front_name = f"{index}_front{front_ext}"
        new_side_name = f"{index}_side{side_ext}"
        shutil.copy(os.path.join(source_folder, files[i]), os.path.join(destination_folder, new_front_name))
        shutil.copy(os.path.join(source_folder, files[i + 1]), os.path.join(destination_folder, new_side_name))
        index += 1

print("Copying and renaming complete. Files are saved in 'train_sorted'.")


Copying and renaming complete. Files are saved in 'train_sorted'.


In [4]:
%pip install pillow-heif

Note: you may need to restart the kernel to use updated packages.


DEPRECATION: Loading egg at c:\users\feder\anaconda3\lib\site-packages\fanogan-0.0.1-py3.12.egg is deprecated. pip 24.3 will enforce this behaviour change. A possible replacement is to use pip for package installation.. Discussion can be found at https://github.com/pypa/pip/issues/12330
DEPRECATION: Loading egg at c:\users\feder\anaconda3\lib\site-packages\torchvision4ad-0.1.2-py3.12.egg is deprecated. pip 24.3 will enforce this behaviour change. A possible replacement is to use pip for package installation.. Discussion can be found at https://github.com/pypa/pip/issues/12330


In [ ]:
import os
import concurrent.futures
from pillow_heif import open_heif
from PIL import Image

folders = [
    r"C:\Thesis\Data\sorted\test_good_sorted",
    r"C:\Thesis\Data\sorted\test_anomaly_sorted",
    r"C:\Thesis\Data\sorted\train_sorted"
]

def convert_heic_to_png(file_path):
    try:
        heif_image = open_heif(file_path)
        image = Image.frombytes(heif_image.mode, heif_image.size, heif_image.data)
        png_path = os.path.splitext(file_path)[0] + ".png"
        image.save(png_path, "PNG")
        print(f"Converted: {os.path.basename(file_path)} → {os.path.basename(png_path)}")
    except Exception as e:
        print(f"Error processing {file_path}: {e}")

heic_files = []
for folder in folders:
    for file in os.listdir(folder):
        if file.lower().endswith(".heic"):
            heic_files.append(os.path.join(folder, file))

with concurrent.futures.ThreadPoolExecutor() as executor:
    executor.map(convert_heic_to_png, heic_files)

print("Conversion complete.")


Converted: 12_side.HEIC → 12_side.png
Converted: 12_front.HEIC → 12_front.png
Converted: 16_side.HEIC → 16_side.png
Converted: 15_side.HEIC → 15_side.png
Converted: 10_side.HEIC → 10_side.png
Converted: 10_front.HEIC → 10_front.png
Converted: 13_front.HEIC → 13_front.png
Converted: 14_side.HEIC → 14_side.png
Converted: 16_front.HEIC → 16_front.png
Converted: 17_front.HEIC → 17_front.png
Converted: 13_side.HEIC → 13_side.png
Converted: 11_side.HEIC → 11_side.png
Converted: 11_front.HEIC → 11_front.png
Converted: 15_front.HEIC → 15_front.png
Converted: 14_front.HEIC → 14_front.png
Converted: 17_side.HEIC → 17_side.png
Converted: 19_side.HEIC → 19_side.png
Converted: 18_front.HEIC → 18_front.png
Converted: 18_side.HEIC → 18_side.png
Converted: 20_side.HEIC → 20_side.png
Converted: 1_side.HEIC → 1_side.png
Converted: 21_front.HEIC → 21_front.png
Converted: 1_front.HEIC → 1_front.png
Converted: 20_front.HEIC → 20_front.png
Converted: 19_front.HEIC → 19_front.png
Converted: 21_side.HEIC → 21

In [ ]:
import os
import shutil
import re

base_output_folder = r"C:\Thesis\Data\MVTEC_Sorted"
input_folders = {
    "train": r"C:\Thesis\Data\sorted\train_sorted",
    "test_good": r"C:\Thesis\Data\sorted\test_good_sorted",
    "test_anomaly": r"C:\Thesis\Data\sorted\test_anomaly_sorted",
}

if not os.path.exists(base_output_folder):
    os.makedirs(base_output_folder)
    print(f"Created base output folder: {base_output_folder}")
else:
    print(f"Base output folder already exists: {base_output_folder}")
filename_pattern = re.compile(r"^(.+?)_\d+_(front|side)\.png$", re.IGNORECASE)
file_count = 0
skipped_files = []

for folder_key, folder_path in input_folders.items():
    if not os.path.exists(folder_path):
        print(f"WARNING: Input folder {folder_path} does not exist. Skipping...")
        continue

    print(f"Processing folder: {folder_path}")
    files_in_folder = os.listdir(folder_path)
    if not files_in_folder:
        print(f"WARNING: No files found in {folder_path}. Skipping...")
        continue

    for file in files_in_folder:
        if file.lower().endswith(".png"):
            match = filename_pattern.match(file)
            if not match:
                skipped_files.append(file)
                continue  

            object_name, view_name = match.groups() 
            object_folder = os.path.join(base_output_folder, object_name)
            view_folder = os.path.join(object_folder, f"{object_name}_{view_name}")
            dataset_subfolder = "train" if folder_key == "train" else os.path.join("test", "good" if folder_key == "test_good" else "anomaly")
            destination_folder = os.path.join(view_folder, dataset_subfolder)
            os.makedirs(destination_folder, exist_ok=True)
            print(f" Created folder (if not exists): {destination_folder}")
            source_path = os.path.join(folder_path, file)
            destination_path = os.path.join(destination_folder, file)

            shutil.copy2(source_path, destination_path) 
            file_count += 1
            print(f" Copied: {file} → {destination_path}")


print(f"\nDataset restructuring complete. {file_count} files copied.")

if skipped_files:
    print("\nSkipped files (filename format may be incorrect):")
    for skipped in skipped_files:
        print(f"- {skipped}")


Base output folder already exists: C:\Thesis\Data\MVTEC_Sorted
📂 Processing folder: C:\Thesis\Data\sorted\train_sorted
📂 Processing folder: C:\Thesis\Data\sorted\test_good_sorted
📂 Processing folder: C:\Thesis\Data\sorted\test_anomaly_sorted

✅ Dataset restructuring complete. 0 files copied.

⚠️ Skipped files (filename format may be incorrect):
- 100_front.png
- 100_side.png
- 101_front.png
- 101_side.png
- 102_front.png
- 102_side.png
- 103_front.png
- 103_side.png
- 104_front.png
- 104_side.png
- 105_front.png
- 105_side.png
- 106_front.png
- 106_side.png
- 107_front.png
- 107_side.png
- 108_front.png
- 108_side.png
- 109_front.png
- 109_side.png
- 10_front.png
- 10_side.png
- 110_front.png
- 110_side.png
- 111_front.png
- 111_side.png
- 112_front.png
- 112_side.png
- 113_front.png
- 113_side.png
- 114_front.png
- 114_side.png
- 115_front.png
- 115_side.png
- 116_front.png
- 116_side.png
- 117_front.png
- 117_side.png
- 118_front.png
- 118_side.png
- 119_front.png
- 119_side.png
- 11

In [ ]:
def convert_masks_to_binary(base_path, output_path=None, threshold=0):

    mask_path = os.path.join(base_path, "ground_truth")

    if output_path:
        os.makedirs(output_path, exist_ok=True)
        os.makedirs(os.path.join(output_path, "good"), exist_ok=True)
        os.makedirs(os.path.join(output_path, "anomaly"), exist_ok=True)
    
    subdirs = ["good", "anomaly"]
    
    print(f"Converting masks in {mask_path} to binary format...")
    

    converted_count = 0
    skipped_count = 0
    error_count = 0
    
    for subdir in subdirs:
        subdir_path = os.path.join(mask_path, subdir)
        if not os.path.exists(subdir_path):
            print(f"Warning: Subdirectory {subdir_path} does not exist")
            continue
            
        print(f"Processing subdirectory: {subdir}")

        for root, _, files in os.walk(subdir_path):
            for file in files:
                if file.lower().endswith(('.png', '.jpg', '.jpeg', '.bmp', '.tif', '.tiff')):
                    mask_file_path = os.path.join(root, file)
                    
                    try:
                        img = Image.open(mask_file_path)
                        if img.mode != 'L':  
                            print(f"  Converting {mask_file_path} from {img.mode} to binary")
                            mask_array = np.array(img)
                            if len(mask_array.shape) == 3:
                                binary_mask = (np.max(mask_array, axis=2) > threshold).astype(np.uint8) * 255
                            else:

                                binary_mask = (mask_array > threshold).astype(np.uint8) * 255
        
                            binary_img = Image.fromarray(binary_mask)
                            if output_path:
                                rel_path = os.path.relpath(mask_file_path, mask_path)
                                save_path = os.path.join(output_path, rel_path)
                                os.makedirs(os.path.dirname(save_path), exist_ok=True)
                            else:
                                save_path = mask_file_path
        
                            binary_img.save(save_path)
                            converted_count += 1
                        else:
                            mask_array = np.array(img)
                            unique_values = np.unique(mask_array)
        
                            if len(unique_values) > 2 or (len(unique_values) == 2 and 
                                                         not (0 in unique_values and 255 in unique_values)):
                                binary_mask = (mask_array > threshold).astype(np.uint8) * 255
                                binary_img = Image.fromarray(binary_mask)
                                if output_path:
                                    rel_path = os.path.relpath(mask_file_path, mask_path)
                                    save_path = os.path.join(output_path, rel_path)
                                    os.makedirs(os.path.dirname(save_path), exist_ok=True)
                                else:
                                    save_path = mask_file_path
                                binary_img.save(save_path)
                                print(f"  Converting grayscale {mask_file_path} to binary")
                                converted_count += 1
                            else:
                                print(f"  Skipped {mask_file_path} (already binary)")
                                skipped_count += 1
                            
                    except Exception as e:
                        print(f"Error processing {mask_file_path}: {e}")
                        error_count += 1
    
    print("\nConversion summary:")
    print(f"  - Converted: {converted_count} files")
    print(f"  - Skipped (already binary): {skipped_count} files")
    print(f"  - Errors: {error_count} files")
    print("Conversion complete!")

In [ ]:
dataset_path = r"C:\Thesis\Data\paperclips_sorted\paperclips_front"
output_path = None
convert_masks_to_rgb(dataset_path, output_path)